# PayGuard — Step 3: Feature Engineering & Preprocessing

Goal of this notebook:
1. Engineer behavioral features (velocity, deviation from history, new-receiver flag, time-of-day, balance consistency)
2. Encode categorical fields
3. Stratified train/test split
4. Scale features
5. Build three imbalance-handling variants of the training set: **SMOTE**, **random undersampling**, and **class-weighting**
6. Save everything so notebook 4 (model training, one model per person) can just load and go

Input: `../data/paysim_upi_adapted.csv` (from notebook 2)

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

np.random.seed(42)
os.makedirs("../data/processed", exist_ok=True)

df = pd.read_csv("../data/paysim_upi_adapted.csv", parse_dates=["timestamp"])
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

Loaded: 80,000 rows x 13 columns


,timestamp,sender_vpa,receiver_vpa,receiver_is_merchant,transaction_mode,amount,sender_balance_before,sender_balance_after,receiver_balance_before,receiver_balance_after,is_fraud,is_flagged_fraud_baseline,fraud_source
0,2026-01-01 01:00:00,user67042315@paytm,user86084959@okhdfcbank,0,CASH_OUT,607616.73,0.0,0.00,1359923.49,2107778.11,0,0,none
1,2026-01-01 01:00:00,user38933223@okaxis,merchant74897456@okaxis,1,PAYMENT,16859.52,0.0,0.00,0.00,0.00,0,0,none
2,2026-01-01 01:00:00,user42891691@okaxis,user92292416@ybl,0,CASH_OUT,45756.23,0.0,0.00,1393202.52,3420103.09,0,0,none
3,2026-01-01 01:00:00,user94748968@ybl,merchant44815218@paytm,1,PAYMENT,4881.52,0.0,0.00,0.00,0.00,0,0,none
4,2026-01-01 01:00:00,user20305914@okicici,user57863579@okicici,0,CASH_IN,104605.78,4968145.5,5072751.28,207302.14,92307.65,0,0,none


## 0. Honest check: how much repeat history do senders actually have?

Behavioral features (velocity, deviation from historical average) only work if a sender appears more than once in the data. PaySim simulates a huge number of distinct accounts, so this is worth checking honestly before building features that assume history exists — this directly ties to our stated assumption that *"sufficient user transaction history is assumed available."*

In [2]:
sender_txn_counts = df["sender_vpa"].value_counts()
print("Senders with only 1 transaction in this sample:", (sender_txn_counts == 1).sum(),
      f"({(sender_txn_counts == 1).mean() * 100:.1f}% of senders)")
print("Senders with 2+ transactions:", (sender_txn_counts >= 2).sum())
print("\nDistribution:")
print(sender_txn_counts.describe())

Senders with only 1 transaction in this sample: 79984 (100.0% of senders)
Senders with 2+ transactions: 8

Distribution:
count    79992.0000
mean         1.0001
std          0.0100
min          1.0000
25%          1.0000
50%          1.0000
75%          1.0000
max          2.0000
Name: count, dtype: float64


**Expected result:** most senders will show up only once in an 80k row sample pulled from a much larger pool of accounts. This is normal and expected — it just means history-dependent features (avg amount so far, velocity) will default to a neutral value for first-time senders, which is itself a realistic real-world case (a brand-new UPI user) rather than a bug. We handle this explicitly below instead of ignoring it.

## 1. Behavioral feature engineering

All of these are computed using only **past** transactions for a given sender (via `.shift()`), never the current row's own values — this avoids leaking the current transaction's own outcome into its own features.

In [3]:
df = df.sort_values(["sender_vpa", "timestamp"]).reset_index(drop=True)

# --- a) Prior transaction count for this sender (0 for a brand-new sender) ---
df["sender_txn_count_so_far"] = df.groupby("sender_vpa").cumcount()

# --- b) Sender's historical average amount, using only PRIOR transactions ---
df["sender_avg_amount_so_far"] = (
    df.groupby("sender_vpa")["amount"]
    .apply(lambda s: s.shift().expanding().mean())
    .reset_index(level=0, drop=True)
)
global_avg_amount = df["amount"].mean()
df["sender_avg_amount_so_far"] = df["sender_avg_amount_so_far"].fillna(global_avg_amount)

# --- c) Amount deviation ratio: how unusual is this amount vs. the sender's own history ---
df["amount_deviation_ratio"] = df["amount"] / df["sender_avg_amount_so_far"].replace(0, np.nan)
df["amount_deviation_ratio"] = df["amount_deviation_ratio"].fillna(1.0)  # no deviation if no history

print("Behavioral history features done.")
df[["sender_vpa", "sender_txn_count_so_far", "sender_avg_amount_so_far", "amount_deviation_ratio"]].head(8)

Behavioral history features done.


,sender_vpa,sender_txn_count_so_far,sender_avg_amount_so_far,amount_deviation_ratio
0,user00001427@ybl,0,182181.324356,0.571514
1,user00001698@okicici,0,182181.324356,0.040384
2,user00004559@ybl,0,182181.324356,0.017063
3,user00004572@okicici,0,182181.324356,0.047814
4,user00005244@okhdfcbank,0,182181.324356,0.548307
5,user00007146@okaxis,0,182181.324356,0.474577
6,user00012426@okaxis,0,182181.324356,0.053080
7,user00012446@okaxis,0,182181.324356,0.090138


In [4]:
# --- d) New-receiver flag: is this the first time this sender has paid this receiver? ---
df["pair_txn_number"] = df.groupby(["sender_vpa", "receiver_vpa"]).cumcount()
df["is_new_receiver"] = (df["pair_txn_number"] == 0).astype(int)

# --- e) Time-of-day features ---
df["hour_of_day"] = df["timestamp"].dt.hour
df["is_off_hours"] = df["hour_of_day"].between(1, 4).astype(int)  # 1 AM - 4 AM window

# --- f) Balance consistency: does sender_balance_after match what we'd expect? ---
# Large mismatches can indicate odd account behavior (a known PaySim fraud signal:
# fraudulent transfers often leave the destination balance at/near zero).
df["sender_balance_error"] = (df["sender_balance_before"] - df["amount"]) - df["sender_balance_after"]
df["receiver_balance_delta"] = df["receiver_balance_after"] - df["receiver_balance_before"]

# --- g) Balance ratio (clean version) ---
df["sender_balance_zero"] = (df["sender_balance_before"] == 0).astype(int)
df["balance_ratio"] = (df["amount"] / df["sender_balance_before"].replace(0, np.nan)).clip(upper=10)
df["balance_ratio"] = df["balance_ratio"].fillna(0)  # zero-balance case is captured by the flag instead

print("Remaining behavioral features done.")
df[["is_new_receiver", "hour_of_day", "is_off_hours", "sender_balance_error",
    "receiver_balance_delta", "balance_ratio"]].describe()

Remaining behavioral features done.


,is_new_receiver,hour_of_day,is_off_hours,sender_balance_error,receiver_balance_delta,balance_ratio
count,80000.0,80000.000000,80000.000000,8.000000e+04,8.000000e+04,80000.000000
mean,1.0,15.313538,0.007137,-2.035886e+05,1.283922e+05,2.128731
std,0.0,4.319550,0.084182,6.580311e+05,8.877731e+05,3.608494
min,1.0,0.000000,0.000000,-6.988673e+07,-1.664863e+06,0.000000
25%,1.0,12.000000,0.000000,-2.495457e+05,0.000000e+00,0.000000
50%,1.0,16.000000,0.000000,-6.804832e+04,0.000000e+00,0.077028
75%,1.0,19.000000,0.000000,-2.893505e+03,1.500236e+05,2.238119
max,1.0,23.000000,1.000000,1.000000e-02,8.270459e+07,10.000000


## 2. Encode categorical fields

`transaction_mode` (CASH_OUT, PAYMENT, CASH_IN, TRANSFER, DEBIT) is one-hot encoded. `receiver_is_merchant` is already binary.

In [5]:
df = pd.get_dummies(df, columns=["transaction_mode"], prefix="mode")
mode_cols = [c for c in df.columns if c.startswith("mode_")]
print("One-hot encoded transaction mode columns:", mode_cols)

One-hot encoded transaction mode columns: ['mode_CASH_IN', 'mode_CASH_OUT', 'mode_DEBIT', 'mode_PAYMENT', 'mode_TRANSFER']


## 3. Assemble the final feature set

Explicitly **excluded** from features: `timestamp`, `sender_vpa`, `receiver_vpa` (identifiers, not predictive signal in raw form — their *behavior* is captured by the engineered features instead), `is_flagged_fraud_baseline` (reference only), and `fraud_source` (this directly encodes the label — including it would be data leakage).

In [6]:
feature_cols = [
    "amount", "receiver_is_merchant",
    "sender_balance_before", "sender_balance_after",
    "receiver_balance_before", "receiver_balance_after",
    "sender_txn_count_so_far", "sender_avg_amount_so_far", "amount_deviation_ratio",
    "is_new_receiver", "hour_of_day", "is_off_hours",
    "sender_balance_error", "receiver_balance_delta", "balance_ratio","sender_balance_zero",
] + mode_cols

X = df[feature_cols].copy()
y = df["is_fraud"].copy()

print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"Label distribution:\n{y.value_counts()}")
X.head()

Feature matrix: 80,000 rows x 21 features
Label distribution:
is_fraud
0    79777
1      223
Name: count, dtype: int64


,amount,receiver_is_merchant,sender_balance_before,sender_balance_after,receiver_balance_before,receiver_balance_after,sender_txn_count_so_far,sender_avg_amount_so_far,amount_deviation_ratio,is_new_receiver,...,is_off_hours,sender_balance_error,receiver_balance_delta,balance_ratio,sender_balance_zero,mode_CASH_IN,mode_CASH_OUT,mode_DEBIT,mode_PAYMENT,mode_TRANSFER
0,104119.20,0,0.00,0.00,1944499.20,2048618.41,0,182181.324356,0.571514,1,...,0,-104119.20,104119.21,0.000000,1,False,True,False,False,False
1,7357.26,1,0.00,0.00,0.00,0.00,0,182181.324356,0.040384,1,...,0,-7357.26,0.00,0.000000,1,False,False,False,True,False
2,3108.50,1,0.00,0.00,0.00,0.00,0,182181.324356,0.017063,1,...,0,-3108.50,0.00,0.000000,1,False,False,False,True,False
3,8710.84,1,27880.38,19169.54,0.00,0.00,0,182181.324356,0.047814,1,...,0,0.00,0.00,0.312436,0,False,False,False,True,False
4,99891.36,0,2860708.27,2960599.63,392173.29,127927.80,0,182181.324356,0.548307,1,...,0,-199782.72,-264245.49,0.034918,0,True,False,False,False,False


## 4. Stratified train/test split

80/20 split, stratified on `is_fraud` so both sets keep the same fraud ratio. **This happens before any resampling** — SMOTE/undersampling must only ever touch the training set, never the test set, or we'd leak information and get an inflated, dishonest evaluation.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]:,} rows, fraud rate {y_train.mean()*100:.4f}%")
print(f"Test:  {X_test.shape[0]:,} rows, fraud rate {y_test.mean()*100:.4f}%")

Train: 64,000 rows, fraud rate 0.2781%
Test:  16,000 rows, fraud rate 0.2812%


## 5. Scale features

We scale **once**, fit only on the training set, and apply that same fitted scaler everywhere (training, test, and every resampled variant below). This keeps a single consistent pipeline rather than juggling separate scaled/unscaled paths per model.

- Logistic Regression and One-Class SVM **need** scaled features (they're distance/gradient-based).
- Random Forest, XGBoost, and Isolation Forest are tree-based and **insensitive** to scaling — feeding them scaled features doesn't hurt them, so one shared pipeline is a reasonable, defensible simplification.

In [8]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("Scaling done. Example means (should be ~0) and std (should be ~1) on the training set:")
print(X_train_scaled.mean().round(2).head())
print(X_train_scaled.std().round(2).head())

Scaling done. Example means (should be ~0) and std (should be ~1) on the training set:
amount                    -0.0
receiver_is_merchant      -0.0
sender_balance_before      0.0
sender_balance_after      -0.0
receiver_balance_before    0.0
dtype: float64
amount                     1.0
receiver_is_merchant       1.0
sender_balance_before      1.0
sender_balance_after       1.0
receiver_balance_before    1.0
dtype: float64


## 6. Build the three imbalance-handling variants

Applied to **supervised models only** (Logistic Regression, Random Forest, XGBoost). Unsupervised models (Isolation Forest, One-Class SVM) don't use resampled/labeled training data at all — they learn from the (scaled) feature distribution directly, which is the whole point of not needing labels.

All three variants are built from `X_train_scaled` / `y_train` — **never** touching the test set.

In [9]:
# --- Variant 1: SMOTE (synthetic oversampling of the minority/fraud class) ---
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print("SMOTE  -> train size:", X_train_smote.shape[0], "| fraud rate:", f"{y_train_smote.mean()*100:.2f}%")

SMOTE  -> train size: 127644 | fraud rate: 50.00%


In [10]:
# --- Variant 2: Random undersampling (reduce the majority/non-fraud class) ---
undersampler = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = undersampler.fit_resample(X_train_scaled, y_train)
print("Undersample -> train size:", X_train_under.shape[0], "| fraud rate:", f"{y_train_under.mean()*100:.2f}%")

Undersample -> train size: 356 | fraud rate: 50.00%


In [11]:
# --- Variant 3: Class-weighting (no resampling — train on original data, penalize fraud misses more) ---
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = dict(zip(classes.tolist(), weights.tolist()))
print("Class weight dict (pass this to model's class_weight param):", class_weight_dict)

Class weight dict (pass this to model's class_weight param): {0: 0.5013945034627558, 1: 179.77528089887642}


## 7. Save everything for notebook 4

One `.pkl` bundle per person to load — no need to re-run feature engineering five times.

In [12]:
bundle = {
    "feature_cols": feature_cols,
    "scaler": scaler,
    # original (unresampled) scaled train/test — used directly by Isolation Forest, One-Class SVM,
    # and as the base for the class-weighting variant of the supervised models
    "X_train_scaled": X_train_scaled,
    "X_test_scaled": X_test_scaled,
    "y_train": y_train,
    "y_test": y_test,
    "class_weight_dict": class_weight_dict,
    # resampled variants (supervised models only)
    "X_train_smote": X_train_smote,
    "y_train_smote": y_train_smote,
    "X_train_under": X_train_under,
    "y_train_under": y_train_under,
}

joblib.dump(bundle, "../data/processed/train_test_variants.pkl")
print("Saved to ../data/processed/train_test_variants.pkl")
print("\nBundle contents:", list(bundle.keys()))

Saved to ../data/processed/train_test_variants.pkl

Bundle contents: ['feature_cols', 'scaler', 'X_train_scaled', 'X_test_scaled', 'y_train', 'y_test', 'class_weight_dict', 'X_train_smote', 'y_train_smote', 'X_train_under', 'y_train_under']


## 8. How to load this in notebook 4 (and your own model notebook)

```python
import joblib
bundle = joblib.load("../data/processed/train_test_variants.pkl")

X_train_scaled = bundle["X_train_scaled"]
y_train = bundle["y_train"]
X_test_scaled = bundle["X_test_scaled"]
y_test = bundle["y_test"]
class_weight_dict = bundle["class_weight_dict"]

# for the SMOTE variant:
X_train_smote, y_train_smote = bundle["X_train_smote"], bundle["y_train_smote"]

# for the undersampling variant:
X_train_under, y_train_under = bundle["X_train_under"], bundle["y_train_under"]
```

Each of the five models should be trained **three times** (once per preprocessing variant) if supervised, or **once** on `X_train_scaled` if unsupervised — then evaluated on the same untouched `X_test_scaled` / `y_test` every time, so results are directly comparable.

## Next: `04_model_training_<your_model>.ipynb` (one per model)

- Logistic Regression
- Random Forest
- XGBoost
- Isolation Forest
- One-Class SVM / LOF

Each trained on the bundle above, evaluated with precision/recall/F1/confusion matrix, then consolidated into one comparison table.